# <center> Taaltheorie en Taalverwerking - 2026 </center>
### <center>  Lab #1 </center>
### <center> Deadline: April 08, 2026 (19:00 CET) </center>
    
***
<center>

Student1 ID: 15847195 \
Student1 Name: Dion Streefkerk

Student2 ID: 123456789 \
Student2 Name: Bob Engel

### **Import Libraries**

In [1]:
import copy
import re

from collections import deque

### **Instructions:**

- This homework is a group assignment. You can work in groups of 2
- The homework is due on the date and time specified above.
- You can submit your homework via Canvas.
- The homework should be submitted in the form of a Jupyter Notebook.
- The name of the notebook should be `Homework1_Student1ID_Student2ID_GroupName.ipynb`
- The notebook should be written in Python 3.
- The notebook should be written in English.
- The notebook should be self-contained, and should be able to be run in one go. This means that all code should be included in the notebook, and the notebook should not depend on any additional files or packages.
- The notebook should be written in a clear way. The code should be well-documented, and the text should be well-written.
- The homework should be your own work. Submission of identical (or near-identical) homework is considered cheating.

### **Overview**

- [Section 1: Regular Expressions](#section-1)
  - [Question 1 (1 point)](#question-1)
  - [Question 2 (1 point)](#question-2)
- [Section 2: Finite State Automata](#section-2)
  - [Section 2.1: Introduction to FSMs](#section-2.1)
  - [Section 2.2: Working with the FSM class](#section-2.2)
    - [Question 3 (2 points)](#question-3)
    - [Question 4 (2 points)](#question-4)
  - [Section 2.3: Defining your own FSM](#section-2.3)
    - [Question 5 (2 points)](#question-5)
  - [Section 2.4: Testing your FSM](#section-2.4)
    - [Question 6 (4 points)](#question-6)
    - [Question 7 (2 point)](#question-7)
  - [Section 2.5: Checking for determinism](#section-2.5)
    - [Question 8 (3 points)](#question-8)
  - [Section 2.6: Defining a FSM for a verb](#section-2.6)
    - [Question 9 (2 points)](#question-9)
    - [Question 10 (1 point)](#question-10)

<a id="section-1"></a>
### **Section 1: Regular Expressions**

<a id="question-1"></a>
#### <font color='#FF0000'>Question 1 (1 point)</font>

Using the Regular Expression (RE) library in Python, write a function **is_valid_phone_number** that takes a string as input and returns True if the string is a valid phone number and False otherwise. A valid phone number is defined as follows:

- It must have the format `+CC-NNNNNNNNN`, where `CC` is the country code and `NNNNNNNNN` is the phone number.
- The `CC` part must contain exactly two digits.
- The `NNNNNNNNN` part must contain exactly 9 digits.

In [2]:
def is_valid_phone_number(phone_number):
    """A simple function to check if a phone number is valid."""

    return re.fullmatch(r"\+\d{2}-\d{9}", phone_number) is not None

In [3]:
# The following strings should be accepted:
print(is_valid_phone_number("+91-123456789"))
print(is_valid_phone_number("+31-612345678"))

# The following strings should be rejected:
print(is_valid_phone_number("+91-1234567890"))
print(is_valid_phone_number("1234567890"))
print(is_valid_phone_number("+911234567890"))
print(is_valid_phone_number("91-123456789"))

True
True
False
False
False
False


<a id="question-2"></a>
#### <font color='#FF0000'>Question 2 (1 point)</font>

Using the Regular Expression (RE) library in Python, write a function **is_valid_email** that takes a string as input and returns True if the string is a valid email address and False otherwise. A valid email address is defined as follows:

- It must have the format `username@domain`, where `username` and `domain` are strings that do not contain the character `@`.
- The `username` part must contain at least one character.
- The `domain` part must contain at least three characters.
- The `domain` part must contain at least one `.` character.
- The `domain` part must contain at least one character before the `.` character.
- The `domain` part must contain at least one character after the `.` character.
- The email address may contain no spaces

Note that a fully accurate regular expression for all email addresses might be very long. If you have managed to fulfill the requirements above, first continue working on the next exercises.

In [4]:
def is_valid_email(email):
    """A simple function to check if an email is valid."""

    pattern = r"[A-Za-z0-9]+(?:[._%+-][A-Za-z0-9]+)*@[A-Za-z0-9][A-Za-z0-9.-]*\.[A-Za-z0-9]+"
    return re.fullmatch(pattern, email) is not None

In [5]:
# The following strings should be accepted:
print(is_valid_email("user@domain.n"))
print(is_valid_email("example@sub.domain.com"))
print(is_valid_email("name@d.com"))
print(is_valid_email("first.last@domain.ac.in"))
print(is_valid_email("simple@example.com"))

# The following strings should be rejected:
print(is_valid_email("userdomain.com"))
print(is_valid_email("user@domaincom"))
print(is_valid_email("@domain.com"))
print(is_valid_email("user@.com"))
print(is_valid_email("user@domain."))

print(is_valid_email(".#$@uva.nl"))
print(is_valid_email(".@#%.nl"))
print(is_valid_email("@@@#%.nl"))
print(is_valid_email(".@.h"))

True
True
True
True
True
False
False
False
False
False
False
False
False
False


<a id="section-2"></a>
### **Section 2: Finite State Automata**

<a id="section-2.1"></a>
#### **Section 2.1: Introduction to FSMs**

The objective of this exercise is to implement a system that will allow us to execute and analyse any arbitrary finite state machine (FSM) specified in Python. Recall that an FSM is a tuple $\langle\Sigma,Q,q_0,F,\delta\rangle$, where $\Sigma$ is the alphabet, $Q$ is the set of states, $q_0$ is the initial state, and $F$ is the set of final states. We use a provided Python class to represent both states in $Q$ and symbols in the alphabet $\Sigma$, and we use Python strings to represent tokens/symbols.

In [6]:
class FSM:

    def __init__(self):
        """
        Initializes the FSM with default values.
        """
        # Initial state of the FSM
        self._initial_state = None
        
        # Dictionary to store transitions
        self._transitions = {}
        
        # Set of end states
        self._end_states = set()

    def get_transitions(self):
        """
        Returns a deep copy of the transitions to prevent direct modifications.
        """
        return copy.deepcopy(self._transitions)

    def add_transition(self, from_state, token, to_state):
        """
        Adds a transition from one state to another using a token. Prevents infinite loops.
        """
        from_state, to_state = from_state.upper(), to_state.upper()
        if token == "" and from_state == to_state:
            print((from_state, token, to_state), "This would generate an infinite loop, transition has not been added.")
            return False
        self._transitions.setdefault(from_state, set()).add((token, to_state))
        return True

    def remove_transition(self, from_state, token, to_state):
        """
        Removes a specified transition. Notifies if the transition does not exist.
        """

        from_state, to_state = from_state.upper(), to_state.upper()
        
        try:
            self._transitions[from_state].remove((token, to_state))
            print("Transition ({}, {}, {}) succesfully removed".format(from_state, token, to_state))
            return True
        except KeyError:
            print("Transition ({}, {}, {}) could not be found.".format(from_state, token, to_state))
            return False

    def get_initial_state(self):
        """
        Returns a deep copy of the initial state to prevent direct modifications.
        """
        return copy.deepcopy(self._initial_state)

    def set_initial_state(self, state):
        """
        Sets the initial state of the FSM.
        """
        self._initial_state = state.upper()

    def get_end_states(self):
        """
        Returns a deep copy of the end states to prevent direct modifications.
        """
        return copy.deepcopy(self._end_states)

    def add_end_state(self, state):
        """
        Adds a state to the set of end states.
        """
        self._end_states.add(state.upper())

    def remove_end_state(self, state):
        """
        Removes a state from the set of end states, with notification if it does not exist.
        """
        state = state.upper()
        if state in self._end_states:
            self._end_states.remove(state)
            print(f"State {state} succesfully removed as end state")
            return True
        print(f"State {state}, is not one of the end states, and can not be removed.")
        return False

    def print_details(self):
        """
        Prints the details of the FSM, including initial state, end states, and transitions.
        """
        print("FSM details:\n- Initial state:\n  - {}".format(self._initial_state))
        print("- End states:")
        if len(self._end_states) == 0:
            print("  - None")
        for state in sorted(self._end_states):
            print("  - {}".format(state))
        print("- Transitions: ")
        for state, transitions in self._transitions.items():
            for token, to_state in transitions:
                print(f"  - {state} -> {token} -> {to_state}")

Given this class we can create a new FSM by creating a new instance of the class and specifying the alphabet, the set of states, the initial state, the set of final states, and the transition function $\delta$. The transition function $\delta$ is a dictionary that maps a pair of a state and a symbol to a set of states.

Now we will create a new FSM and add some transitions to it. We will use the following FSM as an example:

<p align="center">
  <img src="https://raw.githubusercontent.com/roanvanblanken/tttv2024/main/fsm1.png" alt="FSM Image"/>
  <br>
  <em>Figure 1: Finite State Machine (FSM) diagram</em>
</p>

We can do this by using the **add\_transition(state1, token, state2)** function of the FSM object:

In [7]:
fsm1 = FSM()
fsm1.add_transition('q0','a','q1')
fsm1.add_transition('q0','b','q1')
fsm1.add_transition('q1','b','q2')
fsm1.add_transition('q1','','q2')
fsm1.add_transition('q1','a','q2')
fsm1.print_details()

FSM details:
- Initial state:
  - None
- End states:
  - None
- Transitions: 
  - Q0 -> a -> Q1
  - Q0 -> b -> Q1
  - Q1 -> b -> Q2
  - Q1 -> a -> Q2
  - Q1 ->  -> Q2


We furthermore use the function **fsm.set\_initial\_state()** to indicate the name of the initial state, and the predicate **fsm.add\_end\_state()** to specify any number of final states (in our example there happens to be just a single final state). For our example, this looks as follows:

In [8]:
fsm1.set_initial_state('q0')
fsm1.add_end_state('q2')
fsm1.print_details()

FSM details:
- Initial state:
  - Q0
- End states:
  - Q2
- Transitions: 
  - Q0 -> a -> Q1
  - Q0 -> b -> Q1
  - Q1 -> b -> Q2
  - Q1 -> a -> Q2
  - Q1 ->  -> Q2


Inside the FSM class/object the variables such as **self.\_initial\_state** are accessed directly, if you want to access these from outside the object, you should use the **get** functions:

In [9]:
# Accessing the internal variables of the FSM object correctly, from the outside
print("- Initial state:")
print("  - {}".format(fsm1.get_initial_state()))

# End states are stored as a set(), resulting in only unique values.
print("- End states:")
end_states = list(fsm1.get_end_states())
for state in end_states:
  print("  - {}".format(state))

# transitions = dictionary of tuples
print("- Transitions:")
transitions = fsm1.get_transitions()
for state in transitions:
  for (token, end_state) in transitions[state]:
    print("  - {} -> {} -> {}".format(state, token, end_state))

- Initial state:
  - Q0
- End states:
  - Q2
- Transitions:
  - Q0 -> a -> Q1
  - Q0 -> b -> Q1
  - Q1 -> b -> Q2
  - Q1 -> a -> Q2
  - Q1 ->  -> Q2


<a id="section-2.2"></a>
#### **Section 2.2: Working with the FSM class**

In class you have learned to always specify the alphabet $\Sigma$ and the set of states $Q$ when defining an FSM. This is important for clarity, but of course this information in principle can also be extracted from the specification of $\delta$ (only states and symbols not involved in any transition cannot be recovered this way).

<a id="question-3"></a>
#### <font color='#FF0000'>Question 3 (2 points)</font>

Write a function **get_all_states(fsm)** to obtain the list of all states used anywhere in the provided FSM object (i.e., all states showing that you have provided via: **fsm.set\_initial()**, **fsm.add\_end\_state()**, and **fsm.add\_transition()**). Your list should be sorted alphabetically and not contain any duplicates.

In [10]:
def get_all_states(fsm):
    """Returns a list of all states in the FSM"""

    states = set()

    if fsm.get_initial_state() is not None:
        states.add(fsm.get_initial_state())

    states.update(fsm.get_end_states())

    for from_state, transitions in fsm.get_transitions().items():
        states.add(from_state)
        for _, to_state in transitions:
            states.add(to_state)

    return sorted(states)

In [11]:
# Output for get_all_states(fsm1) should be: ['Q0, 'Q1', 'Q2']
get_all_states(fsm1)

['Q0', 'Q1', 'Q2']

<a id="question-4"></a>
#### <font color='#FF0000'>Question 4 (2 points)</font>

Next write a function **get\_alphabet(fsm)** to obtain the list of all symbols used anywhere in the provided FSM object (i.e., all symbols showing that you have provided via: **fsm.add\_transition()**). Your list should be sorted alphabetically and not contain any duplicates.

In [12]:
def get_alphabet(fsm):
    """Returns a list of all unique tokens in the FSM"""

    alphabet = set()

    for transitions in fsm.get_transitions().values():
        for token, _ in transitions:
            if token != "":
                alphabet.add(token)

    return sorted(alphabet)

In [13]:
# Output for get_alphabet(fsm1) should be: ['a', 'b']
get_alphabet(fsm1)

['a', 'b']

<a id="section-2.3"></a>
#### **Section 2.3: Defining your own FSM**

To check if your FSM is working correctly, we have provided a function **accepts(fsm, string)** that checks if the provided FSM accepts a given string. This function will print `Accepted` if the FSM accepts the string, and `Not accepted` otherwise.

In [14]:
def accept(fsm, query):
    """Checks if the FSM accepts the query"""

    # Check if the FSM has an initial state and end states
    if fsm.get_initial_state() == None or fsm.get_end_states() == []:
        return None

    # Create a queue to store the possible transitions
    possibilities = deque([[fsm.get_initial_state(), [], ""]])
    results = []

    # Run a simple Breadth-First Search to find all possible transitions
    while len(possibilities) > 0:

        # Get the first item in the list that is acting as a queue.
        (current_state, parsed_states, parsed_txt) = possibilities.popleft()

        # Check if we have reached an end goal succesfully.
        if parsed_txt == query and current_state in fsm.get_end_states():
            results.append((current_state, parsed_states, parsed_txt))

        # Skip current possibilties if no transitions exist, and it is not a solution.
        if current_state not in fsm._transitions:
            continue

        # See if we can reach an end state by using an empty transition.
        if len(parsed_txt) == len(query):
            next_token = ""
        else:
            next_token = query[len(parsed_txt)]

        # Add all possible transitions to the queue.
        current_transitions = fsm.get_transitions()[current_state]
        for (token, new_state) in current_transitions:
            if token == next_token:
                possibilities.append((new_state, parsed_states + [new_state], parsed_txt + next_token))

    print("Accepted" if len(results) > 0 else "Not accepted")

Python does not have enforced backtracking (repeatedly pressing the semicolon key) to generate all strings accepted by the current FSM currently in memory, but for **fsm1** we know that only the following strings should be accepted:

In [15]:
accept(fsm1, "ab")
accept(fsm1, "aa")
accept(fsm1, "a")

Accepted
Accepted
Accepted


And we also know that the following strings should not be accepted:

In [16]:
accept(fsm1, "aaa")
accept(fsm1, "abab")
accept(fsm1, "abababab")

Not accepted
Not accepted
Not accepted


<a id="question-5"></a>
#### <font color='#FF0000'>Question 5 (2 points)</font>

Now create a new function **create_fsm2()** that creates the FSM object **fsm2** based on the figure below. The function should return the FSM object with the name **fsm2**.

<p align="center">
  <img src="https://raw.githubusercontent.com/roanvanblanken/tttv2024/main/fsm2.png" alt="FSM Image"/>
  <br>
  <em>Figure 2: Finite State Machine (FSM) diagram</em>
</p>

In [17]:
def create_fsm2():
    """Creates and returns a new FSM object with the name: fsm2 """

    fsm2 = FSM()
    fsm2.set_initial_state('q0')
    fsm2.add_end_state('q1')

    fsm2.add_transition('q0', 'x', 'q1')
    fsm2.add_transition('q1', 'x', 'q1')
    fsm2.add_transition('q1', 'y', 'q2')
    fsm2.add_transition('q2', 'x', 'q1')

    return fsm2

In [18]:
# Output for accept(fsm2, "xyxx") should be: "Accepted"
fsm2 = create_fsm2()
accept(fsm2, "xyxx")

Accepted


In [19]:
# Output for accept(fsm2, "xxyyxx") should be: "Not accepted"
fsm2 = create_fsm2()
accept(fsm2, "xxyyxx")

Not accepted


<a id="section-2.4"></a>
#### **Section 2.4: Testing your FSM**

We can observe that this new FSM accepts an *infinite* number of strings. In which order it will enumerate them depends on the exact order in which you specify your transitions. As this FSM can generate an infinite number of strings, we can not just test a small list to see if it works correctly. If we want to make sure not to miss any accepted strings, we could try to search the space of all accepted strings in the order of their length.

<a id="question-6"></a>
#### <font color='#FF0000'>Question 6 (4 points)</font>

Implement a function **get_all_accepted_strings(fsm, length)** to collect the list of all strings of a given length accepted by the given FSM. This list should be sorted and not contain any duplicates.

*Hint: Try to look at the documentation of the deque class in the collections module. (https://www.geeksforgeeks.org/deque-in-python/)*

*Note: running this shouldn't take more than approx. 10 seconds. If your algorithm takes longer, you probably did something wrong*

In [20]:
def get_all_accepted_strings(fsm, length):
    """Returns a list of all accepted strings of a certain length"""

    if fsm.get_initial_state() is None:
        return []

    accepted_strings = set()
    queue = deque([(fsm.get_initial_state(), "")])
    visited = {(fsm.get_initial_state(), "")}

    while queue:
        state, current_string = queue.popleft()

        if len(current_string) == length and state in fsm.get_end_states():
            accepted_strings.add(current_string)

        if state not in fsm.get_transitions():
            continue

        for token, next_state in fsm.get_transitions()[state]:
            new_string = current_string + token

            if len(new_string) > length:
                continue

            item = (next_state, new_string)
            if item not in visited:
                visited.add(item)
                queue.append(item)

    return sorted(accepted_strings)

In [21]:
# Output for print(get_all_accepted_strings(fsm2, 3)) should be: ['xxx', 'xyx']
print(get_all_accepted_strings(fsm2, 3))

['xxx', 'xyx']


In [22]:
# Output for print(get_all_accepted_strings(fsm2, 4)) should be: ['xxxx', 'xxyx', 'xyxx']
print(get_all_accepted_strings(fsm2, 4))

['xxxx', 'xxyx', 'xyxx']


In [23]:
# Output for print(get_all_accepted_strings(fsm2, 5)) should be: ['xxxxx', 'xxxyx', 'xxyxx', 'xyxxx', 'xyxyx']
print(get_all_accepted_strings(fsm2, 5))

['xxxxx', 'xxxyx', 'xxyxx', 'xyxxx', 'xyxyx']


<a id="question-7"></a>
#### <font color='#FF0000'>Question 7 (2 points)</font>

How many strings of length 10 are accepted by the FSM **fsm2**? And how many strings of length 20 are accepted by the FSM **fsm2**? Document your answer in the cell below.

#### <font color='yellow'>Answer:</font>

In [24]:
accepted_10 = len(get_all_accepted_strings(fsm2, 10))
accepted_20 = len(get_all_accepted_strings(fsm2, 20))

print(f"Er zijn {accepted_10} geaccepteerde strings van lengte 10.")
print(f"Er zijn {accepted_20} geaccepteerde strings van lengte 20.")

Er zijn 55 geaccepteerde strings van lengte 10.
Er zijn 6765 geaccepteerde strings van lengte 20.


<a id="section-2.5"></a>
#### **Section 2.5: Checking for determinism (3 points)**

<a id="question-8"></a>
#### <font color='#FF0000'>Question 8 (3 points)</font>

Implement a function **is_deterministic(fsm)** that checks if the given FSM is deterministic. A FSM is deterministic if for each state and each symbol in the alphabet there is at most one transition. If the FSM is deterministic, the function should return True, otherwise it should return False.

In [25]:
def is_deterministic(fsm):
    """Checks if the FSM is deterministic"""

    for transitions in fsm.get_transitions().values():
        seen_tokens = set()
        for token, _ in transitions:
            if token == "":
                return False
            if token in seen_tokens:
                return False
            seen_tokens.add(token)

    return True

In [26]:
# Output for print(deterministic(fsm1)) should be: False
print(is_deterministic(fsm1))

False


In [27]:
# Output for print(deterministic(fsm2)) should be: True
print(is_deterministic(fsm2))

True


<a id="section-2.6"></a>
#### **Section 2.6: Defining a FSM for a verb (3 points)**

<a id="question-9"></a>
#### <font color='#FF0000'>Question 9 (2 points)</font>

Finally, implement a function **create_fsm3()** that creates a new FSM object, called **fsm3** and write up a specification of an FSM that defines the present tense inflectional paradigm of the English verb "do".

In [28]:
def create_fsm3():
    """Creates and returns a new FSM object with the name: fsm3"""

    # Deze FSM accepteert de vormen: do, does en doing.
    fsm3 = FSM()
    fsm3.set_initial_state('q0')
    fsm3.add_end_state('q2')
    fsm3.add_end_state('q4')
    fsm3.add_end_state('q7')

    fsm3.add_transition('q0', 'd', 'q1')
    fsm3.add_transition('q1', 'o', 'q2')
    fsm3.add_transition('q2', 'e', 'q3')
    fsm3.add_transition('q3', 's', 'q4')
    fsm3.add_transition('q2', 'i', 'q5')
    fsm3.add_transition('q5', 'n', 'q6')
    fsm3.add_transition('q6', 'g', 'q7')

    return fsm3

In [29]:
# Output for accept(fsm3, "do") should be: "Accepted"
fsm3 = create_fsm3()
accept(fsm3, "do")

Accepted


In [30]:
# Output for accept(fsm3, "does") should be: "Accepted"
fsm3 = create_fsm3()
accept(fsm3, "does")

Accepted


In [31]:
# Output for accept(fsm3, "doing") should be: "Accepted"
fsm3 = create_fsm3()
accept(fsm3, "doing")

Accepted


<a id="question-10"></a>
#### <font color='#FF0000'>Question 10 (1 point)</font>

Is your FSM deterministic or nondeterministic? Document your answer in the cell below.

#### <font color='yellow'>Answer:</font>

In [32]:
print("Mijn FSM is deterministisch.")
print("Vanuit elke state is er per symbool hoogstens een mogelijke overgang.")

Mijn FSM is deterministisch.
Vanuit elke state is er per symbool hoogstens een mogelijke overgang.
